In [ ]:
# =================================================================
# CELL 1 — setup and CSV picker
# =================================================================
%matplotlib widget
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, OptimizeWarning
from scipy.signal import find_peaks
import warnings
import glob
import ipywidgets as widgets
from IPython.display import display
 
np.seterr(over="ignore", divide="ignore", invalid="ignore")
 
 
def bi_power_law(phi, lam_f, lam_c, Df, Dc):
    term_c = (2.0 ** -phi / 2.0 ** -lam_c) ** Dc
    term_f = (2.0 ** -phi / 2.0 ** -lam_f) ** Df
    return 1.0 / (1.0 + term_c + term_f)
 
 
def log_bi_power_law(phi, lam_f, lam_c, Df, Dc):
    model = bi_power_law(phi, lam_f, lam_c, Df, Dc)
    model = np.clip(model, 1e-300, 1e300)
    return np.log10(model)
 
 
csv_files = sorted(glob.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError("No .csv files found in the current folder.")
 
csv_picker = widgets.Dropdown(
    options=csv_files,
    description='CSV file:',
    style={'description_width': 'initial'},
)
display(csv_picker)

In [ ]:
# =================================================================
# CELL 2 — load data using the file selected above
# =================================================================
FULLPATH = csv_picker.value
print(f"Loading: {FULLPATH}")
 
T = pd.read_csv(FULLPATH, skiprows=1)
 
Phi = T.iloc[:, 0].to_numpy(dtype=float) - 0.5
d_mm = T.iloc[:, 1].to_numpy(dtype=float)
massp = T.iloc[:, 2].to_numpy(dtype=float)
 
valid = ~np.isnan(d_mm) & ~np.isnan(massp) & ~np.isnan(Phi)
Phi, d_mm, massp = Phi[valid], d_mm[valid], massp[valid]
 
order = np.argsort(Phi)
Phi, d_mm, massp = Phi[order], d_mm[order], massp[order]
 
num_raw = massp / (d_mm ** 3)
num_raw[~np.isfinite(num_raw)] = 0.0
no_data = num_raw / np.sum(num_raw)
 
N_cum = np.cumsum(no_data)
N_cum = N_cum / N_cum[-1]
 
mask = N_cum > 0
phi_masked = Phi[mask]
N_masked = N_cum[mask]
logN_masked = np.log10(N_masked)
 
print(f"Loaded {FULLPATH}: {len(Phi)} points, Phi range [{Phi.min():.2f}, {Phi.max():.2f}]")
 

In [ ]:
# =================================================================
# CELL 2 — interactive picker
# =================================================================
picked_points = []  # reset each time this cell runs
MAX_CLICKS = 2
 
fig, ax = plt.subplots(figsize=(7, 6))
ax.semilogy(Phi, N_cum, 'ok', markersize=6, label='Data')
ax.set_xlabel(r'$\Phi$', fontsize=13)
ax.set_ylabel(r'$N(\geq z)/N_0$', fontsize=13)
ax.set_title('Click each visible crossover / kink (1 or 2)')
ax.set_xlim(-8, 16)
ax.invert_xaxis()
ax.set_ylim(1e-18, 100)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=8)
 
colors = ['purple', 'orange']
 
 
def onclick(event):
    if event.inaxes != ax or len(picked_points) >= MAX_CLICKS:
        return
    phi_click = event.xdata
    picked_points.append(phi_click)
 
    i = len(picked_points) - 1
    ax.axvline(phi_click, color=colors[i], linestyle='--', lw=1.5)
    ax.annotate(f'crossover {i+1}\n$\\Phi$={phi_click:.2f}',
                xy=(phi_click, 1e-9), xytext=(5, 0),
                textcoords='offset points', color=colors[i], fontsize=9)
    fig.canvas.draw_idle()
 
    print(f"Crossover {i+1} picked: Phi={phi_click:.3f}")
    if len(picked_points) == MAX_CLICKS:
        print("Max clicks reached. Run the next cell to fit, "
              "or re-run this cell to reset.")
 
 
cid = fig.canvas.mpl_connect('button_press_event', onclick)
plt.tight_layout()
plt.show()
 

In [ ]:
# =================================================================
# CELL 3 — build p0 from clicks and fit
# =================================================================
if len(picked_points) < 1:
    raise RuntimeError(
        "No click registered yet. Click at least once on the Cell 2 "
        "plot at a visible crossover before running this cell."
    )
 
# offset_phi: how far outward (in Phi) to seed lambda_c/lambda_f
# from your click(s). Widen it if fits still fail to converge.
offset_phi = 2.0
 
if len(picked_points) == 1:
    # Single bend: seed lambda_c/lambda_f symmetrically outward from it.
    phi_crossover_click = picked_points[0]
    lamc_guess = phi_crossover_click - offset_phi  # coarse: smaller Phi
    lamf_guess = phi_crossover_click + offset_phi  # fine:   larger Phi
    print(f"1 crossover clicked at Phi={phi_crossover_click:.3f}")
else:
    # Two bends: the left click marks where the coarse regime gives
    # way (seed lambda_c just outside it, to the coarse/smaller-Phi
    # side), and the right click marks where the fine regime takes
    # over (seed lambda_f just outside it, to the fine/larger-Phi side).
    phi_left, phi_right = sorted(picked_points)
    lamc_guess = phi_left - offset_phi
    lamf_guess = phi_right + offset_phi
    print(f"2 crossovers clicked at Phi={phi_left:.3f} and Phi={phi_right:.3f}")
 
Df0_guess = 3.0
Dc0_guess = 3.0
 
p0 = [lamf_guess, lamc_guess, Df0_guess, Dc0_guess]
print(f"Initial guess: lambda_f={p0[0]:.3f}, lambda_c={p0[1]:.3f}, "
      f"Df={p0[2]:.3f}, Dc={p0[3]:.3f}")
 
lower = [-20, -20, 0.1, 0.1]
upper = [20, 20, 15, 15]
p0 = list(np.clip(p0, lower, upper))
 
# Multistart around the click-derived guess: a guess built from one
# or two clicks is still approximate, so try several perturbed
# versions and keep whichever converges with the lowest residual.
rng = np.random.default_rng(0)
n_trials = 20
best_fval = np.inf
p_fit = None
 
for trial in range(n_trials):
    if trial == 0:
        p0_trial = p0
    else:
        factors = rng.uniform(0.5, 1.8, size=2)  # for Df, Dc
        offset_trial = offset_phi * rng.uniform(0.4, 2.0)
        if len(picked_points) == 1:
            lamc_t = np.clip(phi_crossover_click - offset_trial, lower[1], upper[1])
            lamf_t = np.clip(phi_crossover_click + offset_trial, lower[0], upper[0])
        else:
            lamc_t = np.clip(phi_left - offset_trial, lower[1], upper[1])
            lamf_t = np.clip(phi_right + offset_trial, lower[0], upper[0])
        Df_t = np.clip(Df0_guess * factors[0], lower[2], upper[2])
        Dc_t = np.clip(Dc0_guess * factors[1], lower[3], upper[3])
        p0_trial = [lamf_t, lamc_t, Df_t, Dc_t]
 
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", OptimizeWarning)
            popt_trial, pcov_trial = curve_fit(
                log_bi_power_law, phi_masked, logN_masked,
                p0=p0_trial, method="lm", maxfev=10000,
            )
    except (RuntimeError, OptimizeWarning):
        continue
 
    resid_trial = log_bi_power_law(phi_masked, *popt_trial) - logN_masked
    if not np.all(np.isfinite(resid_trial)):
        continue
    fval_trial = np.sum(resid_trial ** 2)
 
    if fval_trial < best_fval:
        best_fval = fval_trial
        p_fit = popt_trial
 
if p_fit is None:
    raise RuntimeError(
        "All multistart attempts failed to converge from this click. "
        "Try re-clicking closer to the visible bend(s), or increase "
        "offset_phi / n_trials above."
    )
 
fval = best_fval
lambda_f, lambda_c, Df, Dc = p_fit
print(f"\nBi-power-law fit (best of {n_trials} multistart attempts from your click(s)):")
print(f"  lambda_f = {lambda_f:.4g} Phi")
print(f"  lambda_c = {lambda_c:.4g} Phi")
print(f"  Df       = {Df:.3f}")
print(f"  Dc       = {Dc:.3f}")
print(f"  log10 RMSE^2 = {fval:.4f}")
 

In [ ]:
# =================================================================
# CELL 4 — plot the fit result
# =================================================================
Phi_fine = np.linspace(Phi.min() - 1, Phi.max() + 1, 500)
N_fit = bi_power_law(Phi_fine, *p_fit)

fig2, ax2 = plt.subplots(figsize=(6, 6), facecolor="w")

ax2.semilogy(Phi_fine, N_fit, "-r", linewidth=1.5, label="Bi-power-law fit")
ax2.semilogy(Phi, N_cum, "ok", markerfacecolor="k", markersize=6,
             linewidth=1, label="Data")

# --- Mark transition regions via curvature, same approach as before ---
logN_fit = np.log10(np.clip(N_fit, 1e-300, None))
dlogN = np.gradient(logN_fit, Phi_fine)
d2logN = np.gradient(dlogN, Phi_fine)

peaks, properties = find_peaks(
    np.abs(d2logN), prominence=np.max(np.abs(d2logN)) * 0.05
)

if len(peaks) >= 2:
    top2 = peaks[np.argsort(properties["prominences"])[-2:]]
    top2 = np.sort(top2)
    phi_crossover1, phi_crossover2 = Phi_fine[top2]
    ax2.axvline(phi_crossover1, color="purple", linestyle=":", linewidth=1.5,
                label=fr"crossover 1: $\Phi$={phi_crossover1:.2f}")
    ax2.axvline(phi_crossover2, color="orange", linestyle=":", linewidth=1.5,
                label=fr"crossover 2: $\Phi$={phi_crossover2:.2f}")
elif len(peaks) == 1:
    phi_crossover1 = Phi_fine[peaks[0]]
    ax2.axvline(phi_crossover1, color="purple", linestyle=":", linewidth=1.5,
                label=fr"crossover: $\Phi$={phi_crossover1:.2f}")

ax2.axvline(lambda_c, color="blue", linestyle="--", linewidth=1, alpha=0.6,
            label=fr"$\lambda_c$={lambda_c:.2f}")
ax2.axvline(lambda_f, color="green", linestyle="--", linewidth=1, alpha=0.6,
            label=fr"$\lambda_f$={lambda_f:.2f}")

ax2.set_xlabel(r"$\Phi$", fontsize=14)
ax2.set_ylabel(r"$N(\geq z)/N_0$", fontsize=14)
fname = FULLPATH.split("/")[-1].split("\\")[-1]
ax2.set_title(f"{fname} (click-picked starting guess)", fontsize=12)
ax2.legend(loc="lower right", fontsize=8)
ax2.grid(True)

ax2.set_xlim(-8, 16)
ax2.invert_xaxis()
ax2.set_ylim(1e-18, 100)

plt.tight_layout()
plt.show()

print(FULLPATH)
print(f"\nBi-power-law fit (best of {n_trials} multistart attempts from your click(s)):")
print(f"  lambda_f = {lambda_f:.4g} Phi")
print(f"  lambda_c = {lambda_c:.4g} Phi")
print(f"  Df       = {Df:.3f}")
print(f"  Dc       = {Dc:.3f}")
print(f"  log10 RMSE^2 = {fval:.4f}")